# 2.3 Quick Check: DWD Gridded vs Station Workflow

This notebook is an evaluation, not a full validation study. The aim is merely to check whether DWD's gridded weather products look different enough from our station-based workflow to justify extra download and processing work.

Keeping it lightweight, we need to:

1. look at what DWD offers,
2. spot-check five Brandenburg districts for precipitation and mean temperature in July 2020,
3. take a step back and ask whether the extra gridded work is likely to matter at NUTS-3 scale.

**Data source:** [DWD CDC grids_germany](https://opendata.dwd.de/climate_environment/CDC/grids_germany/) - open access, no auth.

**Decision rule:** If the gridded and station views stay close here, we keep the station workflow for now and only revisit grids if a fuller zonal-mean test looks worthwhile later on.

In [ ]:
import io
import re
import zipfile
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd
import rasterio
import requests
from rasterio.warp import transform as rio_transform

# Local storage
CACHE_DIR = Path("_cache")
CACHE_DIR.mkdir(exist_ok=True)
GRIDDED_DIR = CACHE_DIR / "dwd_gridded"
GRIDDED_DIR.mkdir(parents=True, exist_ok=True)

# Remote source
DB_PATH = CACHE_DIR / "agri_weather_yield.duckdb"
BASE_URL = "https://opendata.dwd.de/climate_environment/CDC/grids_germany/"
HYRAS_TEMP_DIR = BASE_URL + "daily/hyras_de/air_temperature_mean/"
HYRAS_PRECIP_DIR = BASE_URL + "daily/hyras_de/precipitation/"
REGNIE_INFO_URL = BASE_URL + "daily/regnie/regnie_info.html"
DWD_KL_HIST_DIR = "https://opendata.dwd.de/climate_environment/CDC/observations_germany/climate/daily/kl/historical/"

# HYRAS temperature is stored in tenths of degrees C in this product file.
HYRAS_TEMP_SCALE_FACTOR = 0.1

# Evaluation basis: July 2020
EVAL_YEAR = 2020
EVAL_MONTH = 7

In [6]:
con = duckdb.connect(str(DB_PATH))
con.load_extension("spatial")

# Check basis tables exist
relations: set[str] = {name for (name,) in con.sql("show tables").fetchall()}
required_relations = {"nuts3", "dwd_stations_nuts3", "vw_dwd_station_density_nuts3"}

# Assert that all required tables/views (relations) are present in the db.
missing_relations = required_relations - relations
assert not missing_relations, f"Run eval-dwd-stations-nuts3.ipynb first to create the shared station-density view. Missing: {', '.join(missing_relations)}"

## 1. Look at the DWD grid catalogue

First make sure the products we care about are actually there an can be accessed easily.


In [7]:
catalogue_resp: requests.Response = requests.get(BASE_URL, timeout=30)
catalogue_resp.raise_for_status()

product_links = re.findall(r'href="([^"]+/?)"', catalogue_resp.text)
sorted(
    {  # set
        slug for link in product_links if (slug := link.rstrip("/")) != ".." and not link.startswith("/") and not link.startswith("http")
    }
)

['5_minutes',
 'annual',
 'daily',
 'halfyear',
 'hourly',
 'monthly',
 'multi_annual',
 'return_periods',
 'seasonal']

In [8]:
hyras_temp_resp = requests.get(HYRAS_TEMP_DIR, timeout=30)
hyras_temp_resp.raise_for_status()
hyras_precip_resp = requests.get(HYRAS_PRECIP_DIR, timeout=30)
hyras_precip_resp.raise_for_status()

# Reused here and below
hyras_regex = re.compile(r'href="([^"]+\.nc)"')

hyras_temp_entries = hyras_regex.findall(hyras_temp_resp.text)[-5:]
hyras_precip_entries = hyras_regex.findall(hyras_precip_resp.text)[-5:]

pd.DataFrame(
    [
        {
            "dataset": "HYRAS mean temperature",
            "url": HYRAS_TEMP_DIR,
            "sample_entries": "; ".join(hyras_temp_entries),
        },
        {
            "dataset": "HYRAS daily precipitation",
            "url": HYRAS_PRECIP_DIR,
            "sample_entries": "; ".join(hyras_precip_entries),
        },
        {
            "dataset": "REGNIE status page",
            "url": REGNIE_INFO_URL,
            "sample_entries": "Retired; points users to HYRAS-PRE-DE",
        },
    ]
)

,dataset,url,sample_entries
0,HYRAS mean temperature,https://opendata.dwd.de/climate_environment/CD...,tas_hyras_1_2024_v6-1_de.nc; tas_hyras_1_2025_...
1,HYRAS daily precipitation,https://opendata.dwd.de/climate_environment/CD...,pr_hyras_1_2024_v6-1_de.nc; pr_hyras_1_2025_v6...
2,REGNIE status page,https://opendata.dwd.de/climate_environment/CD...,Retired; points users to HYRAS-PRE-DE


## Quick Note on REGNIE

The old REGNIE download path now returns `404` because DWD retired it in January 2022 and points users to `HYRAS-PRE-DE` instead.

Upstream note: https://opendata.dwd.de/climate_environment/CDC/grids_germany/daily/regnie/regnie_info.html


## 2. Pick a small Brandenburg sample

We do not need all of Germany for a first pass. The idea is to use a handful of Brandenburg districts with active station coverage and see whether the gridded values really tell a different story.

To avoid biasing the pilot toward only the most station-dense districts, we take a reproducible hash-based sample of five active-coverage districts.

In [9]:
con.sql("""
    select count(*)                         as districts_with_active_stations
         , min(n_stations_active)           as min_active_stations
         , round(avg(n_stations_active), 1) as avg_active_stations
    from vw_dwd_station_density_nuts3
    where nuts3_id like 'DE4%'
      and n_stations_active > 0
""").df()

,districts_with_active_stations,min_active_stations,avg_active_stations
0,15,1,1.8


In [10]:
sample_regions = con.sql("""
    select n.nuts_id   as "nuts_id"
         , n.nuts_name as "nuts_name"
         , round(st_x(st_centroid(n.geometry)), 4) as lon_centroid
         , round(st_y(st_centroid(n.geometry)), 4) as lat_centroid
         , d.n_stations_active
    from nuts3 n
    join vw_dwd_station_density_nuts3 d on d.nuts3_id = n.nuts_id
    where n.cntr_code = 'DE'
      and n.nuts_id like 'DE4%'
      and d.n_stations_active > 0
    order by hash(n.nuts_id), n.nuts_id
    limit 5
""").df()
sample_regions

,nuts_id,nuts_name,lon_centroid,lat_centroid,n_stations_active
0,DE40C,Oder-Spree,14.2225,52.2502,3
1,DE40E,Potsdam-Mittelmark,12.6845,52.2506,3
2,DE40I,Uckermark,13.8636,53.2068,2
3,DE404,"Potsdam, Kreisfreie Stadt",13.0382,52.4304,1
4,DE40B,Oberspreewald-Lausitz,13.9404,51.6145,1


## 3. Pull a small HYRAS sample

To keep this light, let's sample the centroid pixel for the same five Jul 2020 days from HYRAS precip & HYRAS mean temp. That's enough for a quick check, even though it's not a full zonal mean.

In [11]:
hyras_precip_index = requests.get(HYRAS_PRECIP_DIR, timeout=30)
hyras_precip_index.raise_for_status()
hyras_temp_index = requests.get(HYRAS_TEMP_DIR, timeout=30)
hyras_temp_index.raise_for_status()

hyras_precip_files = sorted(entry for entry in hyras_regex.findall(hyras_precip_index.text) if f"pr_hyras_1_{EVAL_YEAR}_" in entry)
hyras_temp_files = sorted(entry for entry in hyras_regex.findall(hyras_temp_index.text) if f"tas_hyras_1_{EVAL_YEAR}_" in entry)
assert hyras_precip_files, f"No HYRAS daily precipitation file found for {EVAL_YEAR}."
assert hyras_temp_files, f"No HYRAS daily temperature file found for {EVAL_YEAR}."

hyras_precip_file = hyras_precip_files[-1]
hyras_temp_file = hyras_temp_files[-1]
HYRAS_PR_URL = HYRAS_PRECIP_DIR + hyras_precip_file
HYRAS_TEMP_URL = HYRAS_TEMP_DIR + hyras_temp_file
cache_hyras_precip = GRIDDED_DIR / hyras_precip_file
cache_hyras_temp = GRIDDED_DIR / hyras_temp_file

pd.DataFrame(
    [
        {"metric": "precip_mm", "filename": hyras_precip_file, "cache_file": cache_hyras_precip.name},
        {"metric": "tmean_c", "filename": hyras_temp_file, "cache_file": cache_hyras_temp.name},
    ]
)

,metric,filename,cache_file
0,precip_mm,pr_hyras_1_2020_v6-1_de.nc,pr_hyras_1_2020_v6-1_de.nc
1,tmean_c,tas_hyras_1_2020_v6-1_de.nc,tas_hyras_1_2020_v6-1_de.nc


Let's see how big they are...

In [12]:
for url, path in [(HYRAS_PR_URL, cache_hyras_precip), (HYRAS_TEMP_URL, cache_hyras_temp)]:
    if not path.exists():
        response = requests.get(url, timeout=120, stream=True)
        response.raise_for_status()
        with open(path, "wb") as fh:
            for chunk in response.iter_content(1024 * 1024):
                fh.write(chunk)

pd.DataFrame(
    [
        {
            "metric": "precip_mm",
            "cache_file": cache_hyras_precip.name,
            "size_mb": round(cache_hyras_precip.stat().st_size / 1e6, 1),
        },
        {
            "metric": "tmean_c",
            "cache_file": cache_hyras_temp.name,
            "size_mb": round(cache_hyras_temp.stat().st_size / 1e6, 1),
        },
    ]
)

,metric,cache_file,size_mb
0,precip_mm,pr_hyras_1_2020_v6-1_de.nc,51.0
1,tmean_c,tas_hyras_1_2020_v6-1_de.nc,67.7


In [19]:
from typing import Any


def sample_hyras_grid(
    dataset: Any,
    regions: pd.DataFrame,
    dates: pd.DatetimeIndex,
    *,
    scale_factor: float = 1.0,
) -> pd.DataFrame:
    xs = regions["lon_centroid"].tolist()
    ys = regions["lat_centroid"].tolist()

    if dataset.crs and dataset.crs.to_string() != "EPSG:4326":
        xs, ys = rio_transform("EPSG:4326", dataset.crs, xs, ys)

    samples = np.vstack(list(dataset.sample(zip(xs, ys, strict=True)))).astype(float)
    if dataset.nodata is not None:
        samples[samples == dataset.nodata] = np.nan

    band_numbers = dates.dayofyear.to_numpy()
    assert dataset.count >= int(band_numbers.max()), f"Expected full-year HYRAS stack, got {dataset.count} bands."

    values = samples[:, band_numbers - 1].T * scale_factor

    return pd.DataFrame(
        values,
        index=dates,
        columns=regions["nuts_id"].tolist(),
    ).rename_axis("date")

In [18]:
sample_dates = pd.date_range(f"{EVAL_YEAR}-{EVAL_MONTH:02d}-01", periods=5, freq="D")

with rasterio.open(cache_hyras_precip) as hyras_precip:
    hyras_precip_df = sample_hyras_grid(hyras_precip, sample_regions, sample_dates)

with rasterio.open(cache_hyras_temp) as hyras_temp:
    hyras_temp_df = sample_hyras_grid(
        hyras_temp,
        sample_regions,
        sample_dates,
        scale_factor=HYRAS_TEMP_SCALE_FACTOR,
    )

# Guard against unit mismatches when HYRAS metadata conventions change.
valid_hyras_t = hyras_temp_df.stack().dropna()
assert valid_hyras_t.between(-60.0, 60.0).all(), "HYRAS temperature sample is outside plausible Celsius range; check scale factor."

pd.concat(
    {
        "precip_mm": hyras_precip_df,
        "tmean_c": hyras_temp_df.round(1),
    },
    axis=1,
)

precip_mm                         tmean_c                        
               DE40C DE40E DE40I DE404 DE40B   DE40C DE40E DE40I DE404 DE40B
date                                                                        
2020-07-01      17.6  13.7   0.3   8.2  15.1    19.8  19.3  18.6  19.3  22.0
2020-07-02       0.3   0.3   2.8   0.2   1.5    19.6  19.1  18.5  19.5  20.5
2020-07-03       0.0   0.0   0.0   0.0   0.0    19.1  18.5  18.6  18.8  18.6
2020-07-04       0.0   0.0   0.4   0.0   0.0    20.6  20.3  20.0  20.6  21.3
2020-07-05       1.5   1.0   0.6   1.7   0.0    22.8  22.0  21.0  22.0  23.5

## 4. Compare the HYRAS sample with a direct DWD station pull

To keep the ordering clean, we do this inside the notebook itself: pull the matching DWD station files for these districts, average them back to NUTS-3, and compare them with the HYRAS centroid sample. It is still a quick spot check, not a final validation.

In [20]:
sample_ids = sample_regions["nuts_id"].tolist()
sample_ids_sql = ", ".join(f"'{rid}'" for rid in sample_ids)

station_index = requests.get(DWD_KL_HIST_DIR, timeout=30)
station_index.raise_for_status()
hist_zip_by_station = {
    station_id: DWD_KL_HIST_DIR + filename
    for filename, station_id in re.findall(
        r"(tageswerte_KL_(\d{5})_[0-9_a-z]+\.zip)",
        station_index.text,
    )
}

sample_station_map = con.sql(f"""
    select nuts3_id
         , nuts3_name
         , station_id
         , station_name
         , date_from
         , date_to
    from dwd_stations_nuts3
    where nuts3_id in ({sample_ids_sql})
      and date_from <= date '{sample_dates.max():%Y-%m-%d}'
      and date_to >= date '{sample_dates.min():%Y-%m-%d}'
    order by nuts3_id, station_id
""").df()
sample_station_map["station_id"] = sample_station_map["station_id"].astype(str).str.zfill(5)
sample_station_map = sample_station_map.assign(zip_url=lambda d: d["station_id"].map(hist_zip_by_station)).dropna(subset=["zip_url"]).reset_index(drop=True)

sample_station_map[["nuts3_id", "nuts3_name", "station_id", "station_name", "date_from", "date_to"]].head(5)

,nuts3_id,nuts3_name,station_id,station_name,date_from,date_to
0,DE404,"Potsdam, Kreisfreie Stadt",03987,Potsdam,1893-01-01,2026-03-07
1,DE40B,Oberspreewald-Lausitz,02627,Schipkau-Klettwitz,1996-07-01,2026-03-07
2,DE40C,Oder-Spree,03015,Lindenberg,1906-04-01,2026-03-07
3,DE40C,Oder-Spree,06170,Coschen,2000-02-01,2026-03-07
4,DE40C,Oder-Spree,14138,Falkenberg (Grenzschichtmessfe,2009-09-14,2026-03-07


In [21]:
sample_station_map.columns

Index(['nuts3_id', 'nuts3_name', 'station_id', 'station_name', 'date_from',
       'date_to', 'zip_url'],
      dtype='str')

In [27]:
sample_station_cache = GRIDDED_DIR / (f"dwd_station_daily_precip_tmean_{EVAL_YEAR}_{EVAL_MONTH:02d}_{'_'.join(sample_ids)}.csv")

assert not sample_station_map.empty, "No eligible stations found for selected regions and dates."

# Fill cache on first run only
if not sample_station_cache.exists():
    frames = []
    for row in sample_station_map.itertuples(index=False):
        response: requests.Response = requests.get(row.zip_url, timeout=60)
        response.raise_for_status()

        with zipfile.ZipFile(io.BytesIO(response.content)) as zf:
            data_name = next(name for name in zf.namelist() if name.startswith("produkt_klima"))
            station_df = pd.read_csv(
                io.StringIO(zf.read(data_name).decode("latin-1")),
                sep=";",
                skipinitialspace=True,
            )

        station_df.columns = station_df.columns.str.strip()
        station_df = station_df.assign(
            date=pd.to_datetime(station_df["MESS_DATUM"].astype(str), format="%Y%m%d"),
            precip_mm=pd.to_numeric(station_df["RSK"], errors="coerce"),
            tmean_c=pd.to_numeric(station_df["TMK"], errors="coerce"),
            station_id=row.station_id,
            nuts3_id=row.nuts3_id,
        )[["date", "nuts3_id", "station_id", "precip_mm", "tmean_c"]]
        station_df.loc[station_df["precip_mm"] < -900, "precip_mm"] = np.nan
        station_df.loc[station_df["tmean_c"] < -900, "tmean_c"] = np.nan
        frames.append(station_df.loc[station_df["date"].isin(sample_dates)])

    station_daily = pd.concat(frames, ignore_index=True).sort_values(["date", "nuts3_id", "station_id"]).reset_index(drop=True)
    station_daily.to_csv(sample_station_cache, index=False)

In [28]:
station_daily = pd.read_csv(sample_station_cache, parse_dates=["date"])
station_daily.head(10)

,date,nuts3_id,station_id,precip_mm,tmean_c
0,2020-07-01,DE404,3987,8.6,19.2
1,2020-07-01,DE40B,2627,18.0,22.0
2,2020-07-01,DE40C,3015,19.0,19.8
3,2020-07-01,DE40C,6170,14.7,20.6
4,2020-07-01,DE40E,5546,16.3,18.3
5,2020-07-01,DE40E,6253,NaN,NaN
6,2020-07-01,DE40E,6265,6.5,19.5
7,2020-07-01,DE40I,164,5.6,18.9
8,2020-07-01,DE40I,1869,0.1,18.7
9,2020-07-02,DE404,3987,0.2,19.3


In [29]:
nuts3_station = (
    station_daily.groupby(["date", "nuts3_id"], as_index=False)
    .agg(
        station_precip_mm=("precip_mm", "mean"),
        station_tmean_c=("tmean_c", "mean"),
        precip_station_count=("precip_mm", lambda s: s.notna().sum()),
        tmean_station_count=("tmean_c", lambda s: s.notna().sum()),
    )
    .sort_values(["date", "nuts3_id"])
    .reset_index(drop=True)
)

hyras_precip_long = hyras_precip_df.stack().rename("hyras_precip_mm").rename_axis(index=["date", "nuts3_id"]).reset_index()

hyras_temp_long = hyras_temp_df.stack().rename("hyras_tmean_c").rename_axis(index=["date", "nuts3_id"]).reset_index()

comparison = (
    nuts3_station.merge(hyras_precip_long, on=["date", "nuts3_id"], how="inner")
    .merge(hyras_temp_long, on=["date", "nuts3_id"], how="inner")
    .assign(
        delta_precip_mm=lambda d: round(d["hyras_precip_mm"] - d["station_precip_mm"], 1),
        delta_tmean_c=lambda d: round(d["hyras_tmean_c"] - d["station_tmean_c"], 1),
    )
    .sort_values(["date", "nuts3_id"])
    .reset_index(drop=True)
)

comparison[
    [
        "date",
        "nuts3_id",
        "hyras_precip_mm",
        "station_precip_mm",
        "delta_precip_mm",
        "hyras_tmean_c",
        "station_tmean_c",
        "delta_tmean_c",
    ]
].head(10)

,date,nuts3_id,hyras_precip_mm,station_precip_mm,delta_precip_mm,hyras_tmean_c,station_tmean_c,delta_tmean_c
0,2020-07-01,DE404,8.2,8.60,-0.4,19.3,19.20,0.1
1,2020-07-01,DE40B,15.1,18.00,-2.9,22.0,22.00,0.0
2,2020-07-01,DE40C,17.6,16.85,0.8,19.8,20.20,-0.4
3,2020-07-01,DE40E,13.7,11.40,2.3,19.3,18.90,0.4
4,2020-07-01,DE40I,0.3,2.85,-2.5,18.6,18.80,-0.2
5,2020-07-02,DE404,0.2,0.20,0.0,19.5,19.30,0.2
6,2020-07-02,DE40B,1.5,0.80,0.7,20.5,20.40,0.1
7,2020-07-02,DE40C,0.3,0.20,0.1,19.6,19.65,-0.0
8,2020-07-02,DE40E,0.3,0.30,0.0,19.1,18.85,0.2
9,2020-07-02,DE40I,2.8,1.55,1.2,18.5,18.65,-0.1


In [30]:
valid_precip = comparison.dropna(subset=["hyras_precip_mm", "station_precip_mm"]).copy()
valid_temp = comparison.dropna(subset=["hyras_tmean_c", "station_tmean_c"]).copy()

assert not valid_precip.empty, "No valid precipitation pairs available for comparison."
assert not valid_temp.empty, "No valid temperature pairs available for comparison."

precip_r = float(valid_precip["hyras_precip_mm"].corr(valid_precip["station_precip_mm"]))
temp_r = float(valid_temp["hyras_tmean_c"].corr(valid_temp["station_tmean_c"]))
precip_gap = float(valid_precip["delta_precip_mm"].abs().mean())
temp_gap = float(valid_temp["delta_tmean_c"].abs().mean())

# Pilot acceptance thresholds to avoid relying on correlation alone.
PRECIP_R_MIN = 0.90
TEMP_R_MIN = 0.90
PRECIP_GAP_MAX_MM = 2.0
TEMP_GAP_MAX_C = 1.5

pilot_close_enough = precip_r >= PRECIP_R_MIN and temp_r >= TEMP_R_MIN and precip_gap <= PRECIP_GAP_MAX_MM and temp_gap <= TEMP_GAP_MAX_C

comparison_summary = pd.DataFrame(
    [
        {
            "metric": "precipitation",
            "pearson_r": round(precip_r, 3),
            "valid_pairs": int(len(valid_precip)),
            "mean_station_count": round(float(valid_precip["precip_station_count"].mean()), 2),
            "mean_abs_gap_mm": round(precip_gap, 3),
            "mean_abs_gap_c": np.nan,
            "r_threshold": PRECIP_R_MIN,
            "gap_threshold": PRECIP_GAP_MAX_MM,
            "passes_metric_gate": bool(precip_r >= PRECIP_R_MIN and precip_gap <= PRECIP_GAP_MAX_MM),
        },
        {
            "metric": "temperature",
            "pearson_r": round(temp_r, 3),
            "valid_pairs": int(len(valid_temp)),
            "mean_station_count": round(float(valid_temp["tmean_station_count"].mean()), 2),
            "mean_abs_gap_mm": np.nan,
            "mean_abs_gap_c": round(temp_gap, 3),
            "r_threshold": TEMP_R_MIN,
            "gap_threshold": TEMP_GAP_MAX_C,
            "passes_metric_gate": bool(temp_r >= TEMP_R_MIN and temp_gap <= TEMP_GAP_MAX_C),
        },
    ]
).set_index("metric")

comparison_summary

,pearson_r,valid_pairs,mean_station_count,mean_abs_gap_mm,mean_abs_gap_c,r_threshold,gap_threshold,passes_metric_gate
metric,,,,,,,,
precipitation,0.982,25,1.6,0.5,NaN,0.9,2.0,True
temperature,0.987,25,1.6,NaN,0.204,0.9,1.5,True


We're not looking for a perfect match between a grid cell and a station average, only focusing on whether the gridded view seems to change the district-level story in a meaningful way for these sample days.

## 5. Step back to the NUTS-3 scale

The spot check above is only one angle. This second check is simpler and more practical: if Brandenburg districts already have a few active stations each, then a 1 km grid may not move the district average very much once everything is aggregated back up.


In [31]:
area_stats = con.sql("""
    with  nuts3_area as (
        select nuts_id   "nuts_id"
             , nuts_name "nuts_name"
             , round(st_area(st_transform(geometry, 'EPSG:4326', 'EPSG:3035')) / 1e6, 0) as "area_km2"
        from nuts3
        where cntr_code = 'DE'
          and nuts_id like 'DE4%')
    select a.nuts_id
         , a.nuts_name
         , a.area_km2
         , d.n_stations_active
         , case
             when d.n_stations_active > 0 then round(a.area_km2 / d.n_stations_active, 0)
             else null
           end "km2_per_station"
    from nuts3_area a
    join vw_dwd_station_density_nuts3 d on d.nuts3_id = a.nuts_id
""").df()

area_stats.sort_values(["area_km2", "nuts_id"], ascending=[False, True]).head(15)

,nuts_id,nuts_name,area_km2,n_stations_active,km2_per_station
10,DE40I,Uckermark,4806.0,2,2403.0
5,DE40E,Potsdam-Mittelmark,4201.0,3,1400.0
14,DE40D,Ostprignitz-Ruppin,4109.0,3,1370.0
3,DE40C,Oder-Spree,3683.0,3,1228.0
12,DE406,Dahme-Spreewald,3645.0,1,3645.0
9,DE40F,Prignitz,3287.0,1,3287.0
15,DE409,Märkisch-Oderland,3264.0,3,1088.0
6,DE40H,Teltow-Fläming,3203.0,2,1602.0
13,DE40A,Oberhavel,3030.0,2,1515.0
2,DE407,Elbe-Elster,2879.0,2,1440.0


In [32]:
median_area_km2 = float(area_stats["area_km2"].median())
median_km2_per_station = float(area_stats["km2_per_station"].dropna().median())
area_summary = pd.Series(
    {
        "districts_in_scope": int(len(area_stats)),
        "districts_with_active_station": int(area_stats["km2_per_station"].notna().sum()),
        "median_nuts3_area_km2": median_area_km2,
        "median_km2_per_active_station": median_km2_per_station,
        "approx_active_stations_per_region": round(median_area_km2 / median_km2_per_station, 1) if median_km2_per_station else np.nan,
    },
    name="Brandenburg NUTS-3 scale",
)

area_summary

districts_in_scope                     18.0
districts_with_active_station          15.0
median_nuts3_area_km2                2954.5
median_km2_per_active_station        1515.0
approx_active_stations_per_region       2.0
Name: Brandenburg NUTS-3 scale, dtype: float64

### Practical Takeaway

This is supporting context, not proof. The idea is just to see whether the current station network already looks dense enough that gridded weather would be unlikely to add much once everything is averaged back to district level.


## 6. Decision

Keep the final note short: what this pilot suggests today, and what the next step would be if gridded weather ever becomes worth revisiting.


In [33]:
expected_pairs = len(sample_dates) * len(sample_ids)
valid_joint_pairs = int(comparison.dropna(subset=["hyras_precip_mm", "station_precip_mm", "hyras_tmean_c", "station_tmean_c"]).shape[0])
coverage_pct = round(100.0 * valid_joint_pairs / expected_pairs, 1) if expected_pairs else 0.0

decision = "skip" if pilot_close_enough else "revisit"

reason = (
    f"Pilot outcome for Brandenburg ({len(sample_ids)} districts x {len(sample_dates)} days): "
    f"precip r={precip_r:.3f}, mean abs gap={precip_gap:.3f} mm; "
    f"temp r={temp_r:.3f}, mean abs gap={temp_gap:.3f} C; "
    f"joint coverage={coverage_pct:.1f}%. "
    f"Thresholds were r >= {PRECIP_R_MIN:.2f}/{TEMP_R_MIN:.2f} and gaps <= "
    f"{PRECIP_GAP_MAX_MM:.1f} mm / {TEMP_GAP_MAX_C:.1f} C. "
    + (
        "All checks passed, so keeping station workflow for now is reasonable."
        if pilot_close_enough
        else "At least one check failed, so this pilot is not sufficient to dismiss a gridded workflow."
    )
)

{
    "decision": decision,
    "coverage_pct": coverage_pct,
    "reason": reason,
}

{'decision': 'skip',
 'coverage_pct': 100.0,
 'reason': 'Pilot outcome for Brandenburg (5 districts x 5 days): precip r=0.982, mean abs gap=0.500 mm; temp r=0.987, mean abs gap=0.204 C; joint coverage=100.0%. Thresholds were r >= 0.90/0.90 and gaps <= 2.0 mm / 1.5 C. All checks passed, so keeping station workflow for now is reasonable.'}

In [34]:
con.sql("""
    create table if not exists data_source_registry (
        source_name    varchar primary key,
        url            varchar,
        spatial_level  varchar,
        temporal_range varchar,
        variables      varchar,
        auth_required  boolean,
        decision       varchar,
        reason         varchar,
        coverage_pct   float,
        evaluated_at   timestamp default current_timestamp
    )
""")

con.execute(
    """
    insert or replace into data_source_registry values (
        'dwd_gridded_hyras_regnie',
        'https://opendata.dwd.de/climate_environment/CDC/grids_germany/',
        '1km grid centroid sample -> district station aggregate spot-check',
        '1931/1951-present (daily)',
        'HYRAS tas (mean temp C), HYRAS pr (precipitation mm)',
        false,
        $1, $2, $3,
        current_timestamp
    )
    """,
    [decision, reason, coverage_pct],
)

con.sql("""
    select source_name, decision, coverage_pct
    from data_source_registry
    where source_name = 'dwd_gridded_hyras_regnie'
""").df()

,source_name,decision,coverage_pct
0,dwd_gridded_hyras_regnie,skip,100.0


In [35]:
con.close()